# 06 Retrieval eval

Compare three retrieval setups on the same ground-truth questions (`top_k=5`):

| Retriever | What it does |
|-----------|----------------|
| **Hybrid** | Qdrant **dense** (`dense_ids`) + in-memory **minsearch** sparse (`minsearch_ids`), fused with RRF-style scores |
| **Qdrant** | **Dense only** — cosine similarity on `all-MiniLM-L6-v2` embeddings stored in Qdrant |
| **Elasticsearch** | **Sparse / keyword** — BM25-style search on the same chunk index in ES |

Metrics: **Hit@5**, **MRR@5**, **MAP@5**, **nDCG@5**.


In [ ]:
import runpy
from pathlib import Path
import sys
import os
import json

runpy.run_path(str((Path.cwd() / "_bootstrap.py") if (Path.cwd() / "_bootstrap.py").exists() else (Path.cwd() / "notebooks" / "_bootstrap.py")))
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
for p in (PROJECT_ROOT, PROJECT_ROOT / "app"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from dotenv import load_dotenv
from elasticsearch import Elasticsearch
from evaluation.eval_utils import evaluate
from evaluation.config_paths import ground_truth_path
from search.es_search import search_elasticsearch
from search.hybrid_search import hybrid_search
from search.qdrant_search import search_qdrant

load_dotenv()

from IPython.display import HTML, display
from evaluation.retrieval_plot import save_plot, save_plot_html

def print_metrics(name, m, top_k):
    print(f"\n{name}:")
    print(
        f"Hit@{top_k}: {m[f'Hit@{top_k}']:.3f} | "
        f"MRR@{top_k}: {m[f'MRR@{top_k}']:.3f} | "
        f"MAP@{top_k}: {m[f'MAP@{top_k}']:.3f} | "
        f"nDCG@{top_k}: {m[f'nDCG@{top_k}']:.3f}"
    )

gt_path = ground_truth_path()
with open(gt_path, "r", encoding="utf-8") as f:
    gt_raw = json.load(f)
gt = [{"query": row["question"], "doc_id": row["doc_id"]} for row in gt_raw]

top_k = 5
metrics = {}

metrics["Hybrid"] = evaluate(gt, hybrid_search, top_k=top_k, local=True)
print_metrics("Hybrid", metrics["Hybrid"], top_k)

metrics["Qdrant"] = evaluate(gt, search_qdrant, top_k=top_k, local=True)
print_metrics("Qdrant", metrics["Qdrant"], top_k)

es_index = os.getenv("ES_INDEX", "medical_docs")
es_url = os.getenv("ES_LOCAL_URL", "http://localhost:9200")

try:
    es_client = Elasticsearch(es_url, request_timeout=30)
    if es_client.indices.exists(index=es_index):
        metrics["Elasticsearch"] = evaluate(gt, search_elasticsearch, top_k=top_k, local=True)
        print_metrics("Elasticsearch", metrics["Elasticsearch"], top_k)
    else:
        print(f"\nElasticsearch index '{es_index}' not found. Run: python app/ingestion/es_ingest.py")
except Exception as e:
    print(f"\nElasticsearch check failed: {e}")

out_dir = PROJECT_ROOT / "images"
png_path = save_plot(metrics, top_k, out_dir)
print(f"Saved: {png_path}")
_, html_doc = save_plot_html(metrics, top_k, out_dir)
display(HTML(html_doc))

